In [4]:
# fbref_flat_scraper.py
# Single-block Selenium scraper: teams -> players -> matches -> save CSV/JSON
# Requirements: selenium, webdriver-manager, pandas
# pip install selenium webdriver-manager pandas

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import json
import traceback

# ---------------- CONFIG ----------------
BASE_URL = "https://fbref.com/en/comps/9/2024-2025/2024-2025-Premier-League-Stats"
HEADLESS = True                 # set False to see browser for debugging
IMPLICIT_WAIT = 6
EXPLICIT_TIMEOUT = 15
SLEEP_DELAY = 1.2               # polite delay between requests
RESTART_EVERY = 0               # set >0 to restart driver after N teams (0 = disabled)

# Fields to collect
PLAYER_FIELDS = [
    "player", "position", "age", "games", "games_starts", "minutes",
    "minutes_90s", "goals", "assists", "goals_assists", "goals_pens",
    "pens_made", "pens_att", "cards_yellow", "cards_red"
]
MATCH_FIELDS = [
    "date", "start_time", "comp", "round", "dayofweek", "venue", "result",
    "goals_for", "goals_against", "opponent", "xg_for", "xg_against",
    "possession", "attendance", "captain", "formation", "opp_formation", "referee"
]





In [5]:
# ---------------- DRIVER SETUP ----------------
opt = Options()
if HEADLESS:
    opt.add_argument("--headless")
    opt.add_argument("--disable-gpu")
    opt.add_argument("--no-sandbox")
opt.add_argument("--window-size=1920,1200")
opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                 "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=opt)
driver.implicitly_wait(IMPLICIT_WAIT)
wait = WebDriverWait(driver, EXPLICIT_TIMEOUT)

In [6]:
# ---------------- STEP 1: Get team links ----------------
teams = []
try:
    driver.get(BASE_URL)
    wait.until(EC.presence_of_element_located((By.ID, "results2024-202591_overall")))
    container = driver.find_element(By.ID, "results2024-202591_overall")
    anchors = container.find_elements(By.CSS_SELECTOR, "tbody > tr > .left > a")
    for a in anchors:
        name = a.text.strip()
        href = a.get_attribute("href")
        if name and href:
            teams.append({"team_name": name, "team_url": href})
    print(f"Found {len(teams)} teams.")
except Exception as e:
    print("Failed to load teams from main page:", e)
    traceback.print_exc()
    driver.quit()
    raise SystemExit("Stopping — cannot find team list")



Found 20 teams.


In [7]:
# ---------------- PREP OUTPUT CONTAINERS ----------------
all_players = []    # list of dicts (each row has player fields + team + player_url)
all_matches = []    # list of dicts (each row has match fields + team)
teams_summary = []  # list of dicts summary per team



In [ ]:
# ---------------- STEP 2: Loop teams and scrape ----------------
for idx, t in enumerate(teams):
    team_name = t["team_name"]
    team_url = t["team_url"]
    print(f"\n[{idx+1}/{len(teams)}] Scraping team: {team_name}")

    try:
        _ = driver.title  # quick check
    except Exception:
        print("Driver session dead — restarting driver...")
        try:
            driver.quit()
        except:
            pass
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=opt)
        driver.implicitly_wait(IMPLICIT_WAIT)
        wait = WebDriverWait(driver, EXPLICIT_TIMEOUT)

    # optional periodic restart
    if RESTART_EVERY and idx>0 and idx % RESTART_EVERY == 0:
        print("Periodic driver restart...")
        try:
            driver.quit()
        except:
            pass
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=opt)
        driver.implicitly_wait(IMPLICIT_WAIT)
        wait = WebDriverWait(driver, EXPLICIT_TIMEOUT)

    # --- open team page ---
    try:
        driver.get(team_url)
    except Exception as e:
        print(f"Failed to open {team_name} page: {e}")
        continue
    time.sleep(SLEEP_DELAY)
    

  


[1/20] Scraping team: Liverpool

[2/20] Scraping team: Arsenal

[3/20] Scraping team: Manchester City

[4/20] Scraping team: Chelsea

[5/20] Scraping team: Newcastle Utd

[6/20] Scraping team: Aston Villa

[7/20] Scraping team: Nott'ham Forest

[8/20] Scraping team: Brighton

[9/20] Scraping team: Bournemouth

[10/20] Scraping team: Brentford

[11/20] Scraping team: Fulham

[12/20] Scraping team: Crystal Palace

[13/20] Scraping team: Everton

[14/20] Scraping team: West Ham

[15/20] Scraping team: Manchester Utd

[16/20] Scraping team: Wolves

[17/20] Scraping team: Tottenham

[18/20] Scraping team: Leicester City

[19/20] Scraping team: Ipswich Town

[20/20] Scraping team: Southampton


In [18]:
team_name


'Southampton'

In [9]:
  # ---------------- SCRAPE PLAYERS ----------------
players = []
try:
        # table container id sometimes wrapped, use the "div_stats_standard_9" or "stats_standard_9"
        possible_player_ids = ["div_stats_standard_9", "stats_standard_9", "div_stats_standard"]
        table = None
        for pid in possible_player_ids:
            try:
                wait.until(EC.presence_of_element_located((By.ID, pid)))
                table = driver.find_element(By.ID, pid)
                break
            except:
                continue
        if table is None:
            # fallback: try to find by table css that contains data-stat="player"
            try:
                table = driver.find_element(By.CSS_SELECTOR, "table.stats_table")
            except:
                table = None

        if table is None:
            print("⚠️ Players table not found for", team_name)
        else:
            # ensure visible
            driver.execute_script("arguments[0].style.display = 'block';", table)
            time.sleep(0.4)
            rows = table.find_elements(By.CSS_SELECTOR, "tbody > tr")
            for r in rows:
                try:
                    # skip rows without player cell
                    if not r.find_elements(By.CSS_SELECTOR, '[data-stat="player"]'):
                        continue
                    player_data = {}
                    # capture player name and link if present
                    try:
                        player_anchor = r.find_element(By.CSS_SELECTOR, '[data-stat="player"] a')
                        player_data["player"] = player_anchor.text.strip()
                        player_data["player_url"] = player_anchor.get_attribute("href")
                    except:
                        # fallback to plain text
                        try:
                            player_text = r.find_element(By.CSS_SELECTOR, '[data-stat="player"]').text.strip()
                            player_data["player"] = player_text
                            player_data["player_url"] = ""
                        except:
                            continue
                    # other stats
                    for f in PLAYER_FIELDS:
                        if f == "player":
                            continue
                        try:
                            el = r.find_element(By.CSS_SELECTOR, f'[data-stat="{f}"]')
                            player_data[f] = el.text.strip()
                        except:
                            player_data[f] = ""
                    player_data["team"] = team_name
                    players.append(player_data)
                    all_players.append(player_data)
                except Exception:
                    # skip problematic rows
                    continue
            print(f"  → Players scraped: {len(players)}")
except Exception as e:
        print("Error scraping players for", team_name, "-", e)
        traceback.print_exc()

time.sleep(SLEEP_DELAY)

    

  → Players scraped: 38


In [ ]:
# ---------------- SCRAPE MATCHES ----------------
matches = []
try:
        possible_match_ids = ["matchlogs_for", "matchlogs_all", "matchlogs_all_comp", "matchlogs"]
        table = None
        for mid in possible_match_ids:
            try:
                wait.until(EC.presence_of_element_located((By.ID, mid)))
                table = driver.find_element(By.ID, mid)
                break
            except:
                continue
        if table is None:
            # sometimes match logs are in another container; try CSS fallback
            try:
                table = driver.find_element(By.CSS_SELECTOR, "table#matchlogs_for, table.matchlogs")
            except:
                table = None

        if table is None:
            print("⚠️ Matches table not found for", team_name)
        else:
            # ensure visible
            driver.execute_script("arguments[0].style.display = 'block';", table)
            time.sleep(0.4)
            rows = table.find_elements(By.CSS_SELECTOR, "tbody > tr")
            for r in rows:
                try:
                    # skip rows without td cells (notes or separators)
                    if not r.find_elements(By.CSS_SELECTOR, "td"):
                        continue
                    match_data = {}
                    for f in MATCH_FIELDS:
                        try:
                            el = r.find_element(By.CSS_SELECTOR, f'[data-stat="{f}"]')
                            match_data[f] = el.text.strip()
                        except:
                            match_data[f] = ""
                    match_data["team"] = team_name

                    if any(v for k, v in match_data.items() if k != "team"):
                        matches.append(match_data)
                        all_matches.append(match_data)
                except Exception:
                    continue
            print(f"  → Matches scraped: {len(matches)}")
except Exception as e:
        print("Error scraping matches for", team_name, "-", e)
        traceback.print_exc()




  → Matches scraped: 44


In [11]:

    # ---------------- SUMMARY ----------------
teams_summary.append({
        "team": team_name,
        "url": team_url,
        "players_count": len(players),
        "matches_count": len(matches)
    })

print(f"✅ {team_name}: {len(players)} players, {len(matches)} matches scraped")
time.sleep(SLEEP_DELAY)





✅ Southampton: 38 players, 44 matches scraped


In [12]:
# ---------------- STEP 3: SAVE FILES ----------------
try:
    pd.DataFrame(all_players).to_csv("players.csv", index=False)
    pd.DataFrame(all_matches).to_csv("matches.csv", index=False)
    with open("teams_summary.json", "w", encoding="utf-8") as f:
        json.dump(teams_summary, f, ensure_ascii=False, indent=2)
    print("\nSaved files: players.csv, matches.csv, teams_summary.json")
except Exception as e:
    print("Failed to save outputs:", e)
    traceback.print_exc()


Saved files: players.csv, matches.csv, teams_summary.json


In [13]:
# ---------------- CLEANUP ----------------
try:
    driver.quit()
except:
    pass

In [14]:

# ---------------- FINAL SUMMARY PRINT ----------------
print("\n=== TEAMS SUMMARY ===")
for s in teams_summary:
    print(f"{s['team']}: players={s['players_count']} matches={s['matches_count']}")


=== TEAMS SUMMARY ===
Southampton: players=38 matches=44


In [15]:
import pandas as pd
from sqlalchemy import create_engine,text
DB_USER = "postgres"
DB_PASSWORD = "root"
DB_HOST = "localhost"     
DB_PORT = "5432"          
DB_NAME = "pr_league"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL, echo=True, future=True)
try:
    with engine.connect() as conn:
        print("Connected to PostgreSQL ")
except Exception as e:
    print("Error details:", str(e))




2025-10-31 14:57:03,153 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-10-31 14:57:03,154 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-10-31 14:57:03,159 INFO sqlalchemy.engine.Engine select current_schema()
2025-10-31 14:57:03,160 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-10-31 14:57:03,165 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-10-31 14:57:03,167 INFO sqlalchemy.engine.Engine [raw sql] {}
Connected to PostgreSQL 


In [16]:
players = pd.read_csv("players.csv", encoding="utf-8").fillna("")
matches = pd.read_csv("matches.csv", encoding="utf-8").fillna("")
teams = pd.read_csv("teams_players.csv", encoding="utf-8").fillna("")

players['player'] = players['player'].str.strip()
teams['nomequipe'] = teams['team'].str.strip()


FileNotFoundError: [Errno 2] No such file or directory: 'teams_players.csv'

In [19]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, ForeignKey, Enum, Date, Time

metadata = MetaData()


competition = Table(
    "competition", metadata,
    Column("idcompetition", Integer, primary_key=True, autoincrement=True),
    Column("nomcompetition", String(100), unique=True, nullable=False)
)


saison = Table(
    "saison", metadata,
    Column("id_saison", Integer, primary_key=True, autoincrement=True),
    Column("annee", String(20), nullable=False)
)


equipe = Table(
    "equipe", metadata,
    Column("idequipe", Integer, primary_key=True, autoincrement=True),
    Column("nomequipe", String(100), nullable=False),
    Column("url", String),
    Column("idcompetition", Integer, ForeignKey("competition.idcompetition", ondelete="CASCADE")),
    Column("idsaison", Integer, ForeignKey("saison.id_saison", ondelete="CASCADE")),
)


joueur = Table(
    "joueur", metadata,
    Column("idjoueur", Integer, primary_key=True, autoincrement=True),
    Column("nomjoueur", String(100), nullable=False),
    Column("position", String(50)),
    Column("nationalite", String(50)),
    Column("id_equipe", Integer, ForeignKey("equipe.idequipe", ondelete="CASCADE"))
)


match_table = Table(
    "match", metadata,
    Column("idmatch", Integer, primary_key=True, autoincrement=True),
    Column("date_match", Date),
    Column("heure", Time),
    Column("round", String(50)),
    Column("venue", String(100)),
    Column("idteamhome", Integer, ForeignKey("equipe.idequipe")),
    Column("idteamaway", Integer, ForeignKey("equipe.idequipe")),
    Column("id_competition", Integer, ForeignKey("competition.idcompetition")),
    Column("id_saison", Integer, ForeignKey("saison.id_saison"))
)


resultatmatch = Table(
    "resultatmatch", metadata,
    Column("idresultat", Integer, primary_key=True, autoincrement=True),
    Column("idmatch", Integer, ForeignKey("match.idmatch")),
    Column("idequipe", Integer, ForeignKey("equipe.idequipe")),
    Column("butsmarques", Integer),
    Column("butsconcedes", Integer),
    Column("resultat", Enum("Victoire", "Défaite", "Nul", name="resultat_enum"))
)


statistiquejoueur = Table(
    "statistiquejoueur", metadata,
    Column("idstats", Integer, primary_key=True, autoincrement=True),
    Column("idjoueur", Integer, ForeignKey("joueur.idjoueur")),
    Column("buts", Integer),
    Column("passesdecisives", Integer),
    Column("nbmatchesplayed", Integer),
    Column("cartonsjaunes", Integer),
    Column("cartonsrouges", Integer)
)


metadata.create_all(engine)


2025-10-31 15:49:57,341 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-31 15:49:57,354 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-10-31 15:49:57,355 INFO sqlalchemy.engine.Engine [generated in 0.00127s] {'table_name': 'competition', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-10-31 15:49:57,375 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.